<center><img src="https://github.com/DACSS-CSSmeths/guidelines/blob/main/pics/small_logo_ccs_meths.jpg?raw=true" width="700"/></center>

_____
<a id='home'></a>


# Introduction to Optimization for Decision Making


# Part 3: Homework Adaptation

`data.xlsx` contains the data for all the columns.

In [ ]:
localFile = 'data.xlsx'

In [78]:
import pandas as pd

pairwise_criteria=pd.read_excel(localFile,sheet_name='criteria', index_col=0)
pairwise_criteria

# opening the comparissons
pairwise_learning=pd.read_excel(localFile,sheet_name='learning', index_col=0)
pairwise_friends=pd.read_excel(localFile,sheet_name='friends', index_col=0)
pairwise_school_life=pd.read_excel(localFile,sheet_name='school_life', index_col=0)
pairwise_vocational=pd.read_excel(localFile,sheet_name='vocational_training', index_col=0)
pairwise_college_prep=pd.read_excel(localFile,sheet_name='college_preparation', index_col=0)
pairwise_music=pd.read_excel(localFile,sheet_name='music_classes', index_col=0)
pairwise_criteria=pd.read_excel(localFile,sheet_name='criteria', index_col=0)

Checking some of the comparisions of the criteria, just in case:

In [79]:
pairwise_music

,Lower_Merion,Haverford,Princeton
Lower_Merion,1,0.3,1
Haverford,3,1.0,3
Princeton,1,0.3,1


# The Data Preparation

Since we dont need missing values (nan), the below function removes the missing (nan) values

In [80]:
import numpy as np 
import networkx as nx

def remove_nans_comparisions(nx_pd_adj):
    return {(e[0],e[1]):e[2]['weight'] for e in nx_pd_adj.edges(data=True) if np.isfinite(e[2]['weight'])}

In [81]:
G_learning = nx.from_pandas_adjacency(pairwise_learning,create_using=nx.MultiDiGraph())
G_friends = nx.from_pandas_adjacency(pairwise_friends,create_using=nx.MultiDiGraph())
G_school = nx.from_pandas_adjacency(pairwise_school_life,create_using=nx.MultiDiGraph())
G_vocational = nx.from_pandas_adjacency(pairwise_vocational,create_using=nx.MultiDiGraph())
G_college = nx.from_pandas_adjacency(pairwise_college_prep,create_using=nx.MultiDiGraph())
G_music = nx.from_pandas_adjacency(pairwise_music,create_using=nx.MultiDiGraph())
G_CRIT = nx.from_pandas_adjacency(pairwise_criteria,create_using=nx.MultiDiGraph())

We wil use that code to get all the comparissons:

In [82]:
learning_comparisions = remove_nans_comparisions(G_learning)
friends_comparisions = remove_nans_comparisions(G_friends)
school_comparisions = remove_nans_comparisions(G_school)
vocational_comparisions = remove_nans_comparisions(G_vocational)
college_comparisions = remove_nans_comparisions(G_college)
music_comparisions = remove_nans_comparisions(G_music)
criteria_comparisons = remove_nans_comparisions(G_CRIT)

In [83]:
# take a look at select comparisions
[learning_comparisions, criteria_comparisons]

[{('Lower_Merion', 'Lower_Merion'): 1.0,
  ('Lower_Merion', 'Haverford'): 0.3,
  ('Lower_Merion', 'Princeton'): 0.3,
  ('Haverford', 'Lower_Merion'): 3.0,
  ('Haverford', 'Haverford'): 1.0,
  ('Haverford', 'Princeton'): 1.0,
  ('Princeton', 'Lower_Merion'): 3.0,
  ('Princeton', 'Haverford'): 1.0,
  ('Princeton', 'Princeton'): 1.0},
 {('learning', 'learning'): 1.0,
  ('learning', 'friends'): 6.0,
  ('learning', 'school_life'): 0.75,
  ('learning', 'vocational_training'): 6.75,
  ('learning', 'college_preparation'): 1.0,
  ('learning', 'music_classes'): 4.0,
  ('friends', 'learning'): 0.16,
  ('friends', 'friends'): 1.0,
  ('friends', 'school_life'): 0.125,
  ('friends', 'vocational_training'): 1.125,
  ('friends', 'college_preparation'): 0.16,
  ('friends', 'music_classes'): 0.6,
  ('school_life', 'learning'): 1.3,
  ('school_life', 'friends'): 8.0,
  ('school_life', 'school_life'): 1.0,
  ('school_life', 'vocational_training'): 9.0,
  ('school_life', 'college_preparation'): 1.3,
  ('sc

# AHP

Once installed, we can call the library and use the **Compare** function:

In [84]:
# input each comparisson
import ahpy

learning_comparisions = remove_nans_comparisions(G_learning)
friends_comparisions = remove_nans_comparisions(G_friends)
school_comparisions = remove_nans_comparisions(G_school)
vocational_comparisions = remove_nans_comparisions(G_vocational)
college_comparisions = remove_nans_comparisions(G_college)
music_comparisions = remove_nans_comparisions(G_music)
criteria_comparisons = remove_nans_comparisions(G_CRIT)

learning = ahpy.Compare('learning', learning_comparisions, random_index='saaty')
friends = ahpy.Compare('friends', friends_comparisions, random_index='saaty')
school = ahpy.Compare('school_life', school_comparisions, random_index='saaty')
vocational = ahpy.Compare('vocational_training', vocational_comparisions, random_index='saaty')
college = ahpy.Compare('college_preparation', college_comparisions, random_index='saaty')
music = ahpy.Compare('music_classes', music_comparisions, random_index='saaty')
criteria = ahpy.Compare('criteria', criteria_comparisons, random_index='saaty')

Creating hierarchy:

Remember we have the **hierarchy** Goal <- Criteria <- Alternatives. At this stage, you just need to tell which are the children of the criteria:

In [85]:
criteria.add_children([learning, friends, school, vocational, college, music])

We can see which criterion was more valuable like this:

In [86]:
print(criteria.global_weights)

{'school_life': 0.344, 'learning': 0.2574, 'college_preparation': 0.2551, 'music_classes': 0.0639, 'friends': 0.043, 'vocational_training': 0.0366}


Results:

Now, we know which is the best option:

In [87]:
print(criteria.target_weights)

{'Princeton': 0.4741, 'Haverford': 0.3532, 'Lower_Merion': 0.1728}


8. Assess consistency

The AHP algorithm assumes that when you are comparing you are consistent; but it may detect if you have been inconsistent:

In [88]:
[(val.name,val.consistency_ratio) for val in [learning, friends, school, vocational, college, music, criteria]]

[('learning', 0.0),
 ('friends', 0.0088),
 ('school_life', 0.0),
 ('vocational_training', 0.0088),
 ('college_preparation', 0.0),
 ('music_classes', 0.0012),
 ('criteria', 0.0004)]

We should review comparissons if you get values greater than 0.1 (so, in this case the solution holds).